In [6]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [7]:
file_path = "data_gps/rev1/gps_ags/VOLVO AGUASCALIENTES  DIC 2021.xlsx"
excel = pd.ExcelFile(file_path)
sheets = excel.sheet_names
print(f"Sheets: {sheets}")

Sheets: ['Período Resumen', 'KARLA ZAMUDIO', 'KARLA ZAMUDIO - 2', 'HIACE TRANSPORTE', 'HIACE TRANSPORTE - 2', 'OSCAR ESQUIVEL', 'OSCAR ESQUIVEL - 2', 'HILUX 41', 'HILUX 41 - 2', 'CRV-DANIEL MTZ', 'CRV-DANIEL MTZ - 2', 'MARCH REFACCIONES', 'MARCH REFACCIONES - 2', 'KIA SOUL', 'KIA SOUL - 2', 'RAM RAPID', 'RAM RAPID - 2', 'RAM RESCATES', 'RAM RESCATES - 2', 'FIAT', 'FIAT - 2']


In [8]:
# read each sheet into a dataframe and concatenate them
frames = [] 
for sheet in sheets:
    # skip first sheet and sheets ending with " - 2"
    if sheet.endswith(" - 2") or sheet == sheets[0]:
        continue

    # read each sheet into a dataframe
    df = pd.read_excel(
        file_path, 
        sheet_name=sheet,
        skiprows=4,  # skip first 4 rows (metadata)
        usecols="A:G"
    ) 

    # drop rows where "Inicio de movimiento" is NaN
    df = df.dropna(subset=["Inicio de movimiento"])
    df = df.dropna(subset=["Distancia recorrida total,\nkm"])
    df["unit"] = sheet  # add a column to identify the sheet
    frames.append(df)  

# join all dataframes into one
df_total = pd.concat(frames, ignore_index=True)
print(f"Total rows: {len(df_total)}") # print total number of rows in the final dataframe
print(df_total["unit"].value_counts()) # print count of rows per sheet/unit


Total rows: 1346
unit
OSCAR ESQUIVEL       282
MARCH REFACCIONES    205
RAM RAPID            191
KARLA ZAMUDIO        151
HIACE TRANSPORTE     138
HILUX 41             121
FIAT                 110
RAM RESCATES          63
CRV-DANIEL MTZ        59
KIA SOUL              26
Name: count, dtype: int64


In [9]:
df_total.head()

,Inicio de movimiento,Final de movimiento,"Distancia recorrida total,\nkm",Tiempo de viaje,"Velocidad media,\nkm/h","Max. velocidad,\nkm/h",Tiempo de inactividad,unit
0,"08:41 - Hroes Len, León, Guanajuato, México, 3...","08:52 - México 45, San Isidro De Los Sauces, L...",10.51,0 days 00:11:00,56.0,102.0,0 days 00:05:00,KARLA ZAMUDIO
1,"08:57 - México 45, San Isidro De Los Sauces, L...","09:04 - Carretera León-Irapuato, Silao de la V...",9.94,0 days 00:07:00,85.0,125.0,0 days 00:09:00,KARLA ZAMUDIO
2,"09:13 - México 45, Los Fresnos, Silao de la Vi...","09:23 - [AGENCIA SILAO] México 45, Acapulquito...",11.35,0 days 00:10:00,68.0,111.0,0 days 02:40:00,KARLA ZAMUDIO
3,"12:04 - [AGENCIA SILAO] México 45, Acapulquito...","12:12 - Carretera Irapuato-León, Colonias Nuev...",8.61,0 days 00:07:00,65.0,114.0,0 days 00:10:00,KARLA ZAMUDIO
4,"12:22 - Carretera Irapuato-León, Colonias Nuev...","12:41 - [AGENCIA SILAO] México 45, Acapulquito...",19.80,0 days 00:19:00,63.0,118.0,0 days 01:06:00,KARLA ZAMUDIO


In [11]:
# group by unit and sum the distance
distance_by_unit = df_total.groupby("unit")["Distancia recorrida total,\nkm"].sum()
distance_by_unit

unit
CRV-DANIEL MTZ       2879.09
FIAT                 4055.06
HIACE TRANSPORTE     3866.43
HILUX 41             2685.52
KARLA ZAMUDIO        4513.52
KIA SOUL              115.21
MARCH REFACCIONES    5593.31
OSCAR ESQUIVEL       6833.34
RAM RAPID            9075.86
RAM RESCATES         1098.53
Name: Distancia recorrida total,\nkm, dtype: float64

In [ ]:
# check columns to confirm structure
df_total.columns.tolist()

['Inicio de movimiento',
 'Final de movimiento',
 'Distancia recorrida total,\nkm',
 'Tiempo de viaje',
 'Velocidad media,\nkm/h',
 'Max. velocidad,\nkm/h',
 'Tiempo de inactividad',
 'unit']

In [18]:
# average velocity and velocity max by unit 
group_by_unit = df_total.groupby("unit").agg(
    {
        "Distancia recorrida total,\nkm": "sum",
        "Velocidad media,\nkm/h": "mean",
        "Max. velocidad,\nkm/h": "max"
    }
)
group_by_unit

,"Distancia recorrida total,\nkm","Velocidad media,\nkm/h","Max. velocidad,\nkm/h"
unit,,,
CRV-DANIEL MTZ,2879.09,42.915254,180.0
FIAT,4055.06,42.654545,146.0
HIACE TRANSPORTE,3866.43,35.355072,132.0
HILUX 41,2685.52,41.851240,156.0
KARLA ZAMUDIO,4513.52,49.582781,144.0
KIA SOUL,115.21,16.923077,70.0
MARCH REFACCIONES,5593.31,44.039024,148.0
OSCAR ESQUIVEL,6833.34,38.645390,144.0
RAM RAPID,9075.86,48.387435,166.0


In [20]:
group_by_unit["liters"] = group_by_unit["Distancia recorrida total,\nkm"] / 17.23
group_by_unit["cost"] = group_by_unit["liters"] * 21.5
group_by_unit.sort_values("cost", ascending=False)

,"Distancia recorrida total,\nkm","Velocidad media,\nkm/h","Max. velocidad,\nkm/h",liters,cost
unit,,,,,
RAM RAPID,9075.86,48.387435,166.0,526.747533,11325.071967
OSCAR ESQUIVEL,6833.34,38.645390,144.0,396.595473,8526.802670
MARCH REFACCIONES,5593.31,44.039024,148.0,324.626233,6979.464016
KARLA ZAMUDIO,4513.52,49.582781,144.0,261.957052,5632.076611
FIAT,4055.06,42.654545,146.0,235.348810,5059.999420
HIACE TRANSPORTE,3866.43,35.355072,132.0,224.401045,4824.622461
CRV-DANIEL MTZ,2879.09,42.915254,180.0,167.097504,3592.596344
HILUX 41,2685.52,41.851240,156.0,155.863030,3351.055136
RAM RESCATES,1098.53,37.603175,162.0,63.756820,1370.771619


In [24]:
# limit km/mounth
# traffic light rules
def traffic_light(km):
    if km < 3000:
        return "green"
    elif km <= 6000:
        return "yellow"
    else:
        return "red"

In [25]:
group_by_unit["status"] = group_by_unit["Distancia recorrida total,\nkm"].apply(traffic_light)
group_by_unit.sort_values("Distancia recorrida total,\nkm", ascending=False)

,"Distancia recorrida total,\nkm","Velocidad media,\nkm/h","Max. velocidad,\nkm/h",liters,cost,status
unit,,,,,,
RAM RAPID,9075.86,48.387435,166.0,526.747533,11325.071967,red
OSCAR ESQUIVEL,6833.34,38.645390,144.0,396.595473,8526.802670,red
MARCH REFACCIONES,5593.31,44.039024,148.0,324.626233,6979.464016,yellow
KARLA ZAMUDIO,4513.52,49.582781,144.0,261.957052,5632.076611,yellow
FIAT,4055.06,42.654545,146.0,235.348810,5059.999420,yellow
HIACE TRANSPORTE,3866.43,35.355072,132.0,224.401045,4824.622461,yellow
CRV-DANIEL MTZ,2879.09,42.915254,180.0,167.097504,3592.596344,green
HILUX 41,2685.52,41.851240,156.0,155.863030,3351.055136,green
RAM RESCATES,1098.53,37.603175,162.0,63.756820,1370.771619,green


In [33]:
# rename columns for better readability
group_by_unit = group_by_unit.rename(columns={
    "Distancia recorrida total,\nkm": "distance_km",
    "Velocidad media,\nkm/h": "avg_speed_kmh",
    "Max. velocidad,\nkm/h": "max_speed_kmh"
})

In [34]:
# export to excel
group_by_unit.sort_values("cost", ascending=False).to_excel("fleet_report_ags.xlsx", 
                                                            sheet_name="summary")

In [35]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

In [36]:
# apply traffic light coloring formarting
wb = load_workbook("fleet_report_ags.xlsx")
ws = wb["summary"]

colors = {
    "green": "90EE90",
    "yellow": "FFFF00",
    "red": "FF6347"
}

# status column is in column G, data starts from row 2
for row in range(2, ws.max_row + 1):
    status = ws.cell(row=row, column=7).value # column G is status
    if status in colors:
        fill = PatternFill(start_color=colors[status], fill_type="solid")
        for col in range(1, ws.max_column + 1):
            ws.cell(row=row, column=col).fill = fill

wb.save("fleet_report_ags.xlsx")
print("Applied traffic light coloring formarting")

Applied traffic light coloring formarting
